# Leyenda - Livrable 3 - Captioning

#### Import

## Architectures schématiques

Pour nos architectures GRU et LSTM, nous avons utilisé une architecture de type CNN pour le pré-traitement des images. Nous avons décidé d'utiliser InceptionV3 (avec ImageNet) pour en tant que CNN afin de transformer les images en des représentations numériques informatives (embeddings visuels). Ces représentations sont ensuite utilisées comme entrées pour les modèles de langage (GRU et LSTM) qui génèrent des légendes pour les images.

L'avantage d'utiliser InceptionV3 (avec ImageNet) est de pouvoir identifier à la fois des détails fins mais aussi des motifs globaux. Ils seront ensuite traduits en vecteurs compréhensibles par le modèle de langage. En utilisant un CNN pré-entraîné, nous pouvons tirer parti de la puissance de l'apprentissage profond sans avoir besoin d'un grand ensemble de données d'images pour entraîner notre propre CNN à partir de zéro. Cela nous permet également de bénéficier des connaissances acquises par le modèle sur un large éventail d'images et de classes.

L'objectif principal de cette étape est de compresser les informations visuelles tout en mettant en relief les éléments clés des images. Cela permet de ne pas surcharger la mémoire lorsqu'il y'a des milliers d'images à traiter.

En amont de cela, nous avons effectué le pré-traitement des images, qui consiste à redimensionner les images à une taille fixe (299x299 pixels) et à les normaliser pour que les valeurs des pixels soient comprises entre 0 et 1. Cela permet de garantir que toutes les images ont la même taille et la même échelle de valeurs, ce qui est essentiel pour l'entraînement du modèle.

Pour le pré-traitement du texte, nous avons effectué les étapes suivantes :

- Nettoyage du texte :

        Passage en minuscules, suppression de la ponctuation si nécessaire, nettoyage des caractères spéciaux.

- Ajout de tokens spéciaux :

        Chaque phrase est entourée de deux tokens : <start> (début) et <end> (fin), pour aider le modèle à comprendre quand commencer et arrêter la génération.

- Tokenisation :

        On convertit chaque mot en entier unique, grâce à un dictionnaire (tokenizer) construit sur l’ensemble des légendes du dataset.

- Padding :

        Les séquences (phrases) sont de tailles variables, donc on les complete (padding) avec des zéros pour qu’elles aient toutes la même longueur.

- Embedding :

        Ces entiers sont ensuite convertis en vecteurs denses à l’aide d’une Embedding layer, qui apprend à capturer la signification sémantique des mots.

### Architecture avec GRU

<img src="./../figures/Architecture_captionning_GRU.png" alt="Architecture_captionning_GRU" style="width: 800px;"/>

### Architecture avec LSTM

<img src="./../figures/Architecture_captionning_LSTM.png" alt="Architecture_captionning_LSTM" style="width: 800px;"/>

## Code des modèles de captionning

Nous allons entraîner deux modèles de réseaux de neurones récurrents, LSTM et GRU, sur notre jeu de données de captioning d’images. L’objectif est de comparer leurs performances en termes de qualité des légendes générées, de rapidité d’entraînement et de capacité à généraliser. Cette comparaison nous permettra de déterminer lequel des deux modèles est le plus adapté à notre tâche.

Le modèle GRU (Gated Recurrent Unit) est une version simplifiée du LSTM. Il fusionne certaines portes pour réduire le nombre de paramètres, ce qui le rend plus rapide et moins coûteux en ressources. Il est souvent aussi performant que le LSTM, surtout sur des jeux de données de taille moyenne ou lorsque les séquences ne sont pas trop longues. 

Le modèle LSTM (Long Short-Term Memory) est conçu pour capturer les dépendances à long terme dans les séquences. Il utilise une cellule de mémoire interne et trois portes (oubli, entrée, sortie) pour gérer l'information de manière fine. Il est particulièrement adapté aux séquences longues et complexes, mais peut être plus lent à entraîner en raison de sa structure plus lourde.



### Modèle avec GRU

In [11]:
GRU_Captionning_model = ModelLoader(model_name="GRU_Captionning")

encoder_GRU, decoder_GRU = GRU_Captionning_model.create_captionning_autoencoder(vocab_size=vocab_size, is_gru=True, init_weigths_path_encoder="../models/weights/caption/encoder_captioning_excellent.weights.h5", init_weigths_path_decoder="../models/weights/caption/decoder_captioning_excellent.weights.h5")

### Modèle avec LSTM

In [ ]:
LSTM_Captionning_model = ModelLoader(model_name="LSTM_Captionning")

encoder_LSTM, decoder_LSTM = LSTM_Captionning_model.create_captionning_autoencoder(vocab_size=vocab_size, is_gru=False)

## Analyse des résultats

### Courbes

### Tableau des performances

### Conclusion

## Exemples de légendes

## Pistes d'amélioration

Plusieurs pistes peuvent être envisagées pour améliorer la qualité des légendes générées.


L’utilisation du Beam Search permettrait d’améliorer la stratégie de prédiction. En génération de texte, la méthode la plus simple (dite greedy) consiste à choisir à chaque étape le mot ayant la plus forte probabilité. Cependant, cette approche peut mener à des séquences sous-optimales. Le Beam Search propose d’explorer plusieurs séquences candidates en parallèle (déterminées par un paramètre appelé beam width) et de conserver uniquement les plus prometteuses à chaque étape. Cette stratégie permet de générer des phrases plus cohérentes et souvent plus proches du sens attendu.

L’entraînement du modèle pourrait être optimisé en s’appuyant sur un plus grand volume de données, ou en appliquant de l’augmentation de données à plusieurs reprises, afin d’accroître la diversité visuelle et améliorer la capacité de généralisation du modèle. Aujourd'hui, nous avons testé les modèles avec 6000 images au maximum mais cela n'est pas le plus performant. Dans l'avenir utiliser les 80000 images du dataset MS COCO serait plus intéressant afin d'optimiser notre modèle.

# Leyenda - Livrable 3 - Pipeline

## Initialisation

In [3]:
import os
import tensorflow as tf
import sys

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)

from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths
from src.utils import show_clean_and_noisy_images
from src.utils import build_image_model, load_tokenizer, filter_by_custom_binary_model, clean_invalid_images
    
data_loader = DataLoader()

classification_model_loader = ModelLoader(model_name="classification_Inception")

autoencoder_skiplayer_loader = ModelLoader(model_name="autoencoder_skiplayer")

2025-04-24 22:22:25.516658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745526145.546329   27504 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745526145.555008   27504 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745526145.611147   27504 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745526145.611222   27504 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745526145.611225   27504 computation_placer.cc:177] computation placer alr

## Chargement et traitement des données

In [4]:
directory_to_load = "../datasets/Test"

In [5]:
clean_invalid_images(directory_to_load)

I0000 00:00:1745526375.488605   27504 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1991 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## Pipeline de données

### **1ère étape : Classification photo/pas photo**
Chargement de notre modèle le plus performant : Inception sur le dataset binaire avec class weight

In [7]:
inception_model = classification_model_loader.create_model_with_inception(show_summary=False, init_weigths_path="./../models/weights/inception/binary_cw-20250411-145213.weights.h5")

/home/fares/datascience/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Activation de la fonction pour séparer les photos en utilisant le modèle de classification à partir du dossier Test

In [9]:
filter_by_custom_binary_model(inception_model, directory_to_load, "./../datasets/Photo_filtered", threshold=0.7, max_images=100)

Analyse de 100 images...


  1%|          | 1/100 [00:00<00:17,  5.66it/s]

schematics_04453.jpg → 0.55


  2%|▏         | 2/100 [00:00<00:16,  5.97it/s]

schematics_04451.jpg → 0.00


...

72.png → 0.00


 99%|█████████▉| 99/100 [00:12<00:00,  7.10it/s]

painting_00122.jpg → 0.00


100%|██████████| 100/100 [00:12<00:00,  8.04it/s]

photo_8308.jpg → 1.00
28/100 images conservées dans : ./../datasets/Photo_filtered


### **2ème étape : Traitement des photos**
Création du modèle autoencodeur avec ses poids

In [11]:
denoising_autoencoder = autoencoder_skiplayer_loader.create_skip_layer_autoencoder(show_summary=False, init_weigths_path="../models/weights/autoencoder/autoencoder_skip_layers_mae_tanguy.weights.h5")


Encoder created successfully.
Decoder created successfully.
Autoencoder autoencoder_skiplayer created successfully.


Utilisation du modèle pour traiter les images

In [12]:
photo_denoised_path = "./../datasets/Photo_denoised"
from src.utils import denoise_images

denoise_images(
    input_dir="./../datasets/Photo_filtered",
    output_dir=photo_denoised_path,
    model=denoising_autoencoder,
    target_size=(256, 256) 
)

I0000 00:00:1745526589.038124   28337 service.cc:152] XLA service 0x7efbba2917f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745526589.038729   28337 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2025-04-24 22:29:49.125648: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Image traitée : photo_0002.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Image traitée : photo_0064.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

I0000 00:00:1745526591.111057   28337 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Image traitée : photo_8284.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Image traitée : photo_8300.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Image traitée : photo_8286.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Image traitée : photo_8287.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Image traitée : photo_0078.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Image traitée : photo_0052.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Image traitée : photo_8289.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Image traitée : photo_0016.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Image traitée : photo_8295.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Image traitée : photo_0080.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Image traitée : photo_8267.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Image traitée : photo_8273.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Image traitée : photo_0032.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Image traitée : photo_0073.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/s

### **3 ème Etape : Captionning**

Chargement du tokenizer

In [ ]:
tokenizer_path = "./../models/weights/captioning_token/captioning_tokenizer_excellent.json"
tokenizer = load_tokenizer(tokenizer_path)
vocab_size = tokenizer.num_words + 1

print("Tokenizer loaded | Vocab size:", vocab_size)

tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'

Chargement du pré-traitement (InceptionV3) et de notre meilleur modèle de captionning

In [ ]:
image_features_extract_model = build_image_model()

GRU_Captionning_model = ModelLoader(model_name="GRU_Captionning")

encoder, decoder = GRU_Captionning_model.create_captionning_autoencoder(vocab_size=vocab_size, is_gru=True, init_weigths_path_encoder="./../models/weights/caption/encoder_captioning_excellent.weights.h5", init_weigths_path_decoder="./../models/weights/caption/decoder_captioning_excellent.weights.h5")

Lancement de la prédiction

In [ ]:
# === Prédiction sur un dossier ===
test_folder = "../datasets/Photo_denoised/"
max_length = 50

for fname in os.listdir(test_folder):
    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(test_folder, fname)
        print(f"\n🖼️ {fname}")
        result, attn = captionning_evaluate(img_path, encoder, decoder,max_length, tokenizer, image_features_extract_model=image_features_extract_model)
        print("📝 Caption:", ' '.join(result))
        print("🔎 Token ids generated:", [tokenizer.word_index.get(w, '??') for w in result])
        plot_attention(img_path, result, attn)
